# OCCLUDE pipeline overhaul — A100 baseline + Step 4 validation

**Goal:** confirm the full overhaul (Steps 1-4) runs on the A100 and measure the speedup against H1.

**H1 (post-Step-1, locked):** fps 8.34, GPU util 33.4%, peak VRAM 2.66 GB, hash `396d6e6d1d175e8ef6b37088136f60ca`.

**Targets:** ≥25 fps, util ≥70%, **hash unchanged**.

Every long-running cell below streams status — model loads, JIT compile, frame progress — so you can see exactly what stage you're in instead of staring at a blinking cursor.

## 1. Sanity check — A100

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone latest `pipeline-overhaul`

In [ ]:
%cd /content
!rm -rf OCCLUDE
!git clone --branch pipeline-overhaul --depth 1 https://github.com/anaxoniclabs/OCCLUDE.git
%cd /content/OCCLUDE
!git log --oneline -5

## 3. Install python deps

Each `pip install` here prints its own progress. The whole cell takes ~60s.

In [ ]:
!pip install transformers ultralytics insightface scipy tqdm rich pillow opencv-python
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null
!pip install onnxruntime-gpu
!pip install pynvml
!pip install torchcodec

## 3a. Install ffmpeg with NVENC

Colab's stock `apt install ffmpeg` lacks `h264_nvenc` — without it `cuda_io_available()` returns `False` and Steps 3+4 are unavailable. We grab a BtbN static build that ships nvenc and put it first on `PATH`. ~10 s.

(Earlier this caused a hang because the encode subprocess died on first write while the main thread blocked on a full queue. Both bugs are now fixed: a) `cuda_io_available()` probes for h264_nvenc explicitly, b) the async runner re-checks worker errors every second during `put`.)

In [ ]:
import os, time
print(f'[{time.strftime("%H:%M:%S")}] downloading static ffmpeg with nvenc…', flush=True)
!wget -q --show-progress -O /tmp/ffmpeg.tar.xz https://github.com/BtbN/FFmpeg-Builds/releases/download/latest/ffmpeg-master-latest-linux64-gpl.tar.xz
print(f'[{time.strftime("%H:%M:%S")}] extracting…', flush=True)
!mkdir -p /opt/ffmpeg-nvenc && tar -xJf /tmp/ffmpeg.tar.xz -C /opt/ffmpeg-nvenc --strip-components=1
os.environ['PATH'] = '/opt/ffmpeg-nvenc/bin:' + os.environ['PATH']
print(f'[{time.strftime("%H:%M:%S")}] which ffmpeg →', flush=True)
!which ffmpeg
print(f'\n[{time.strftime("%H:%M:%S")}] encoder probe →', flush=True)
!ffmpeg -hide_banner -encoders 2>&1 | grep -i h264_nvenc || echo '✗ h264_nvenc NOT FOUND — Step 3+4 will fall back'

## 4. Confirm both fast paths fire

In [ ]:
import sys, time
sys.path.insert(0, '/content/OCCLUDE')

print(f'[{time.strftime("%H:%M:%S")}] importing torch + occlude (loads CUDA libs)…', flush=True)
import torch
from occlude.pipeline.video import _BLUR_DEVICE, _default_perception_batch
from occlude.pipeline.io_cuda import (
    cuda_io_available, _ffmpeg_has_h264_nvenc, _TORCHCODEC_OK,
)
print(f'[{time.strftime("%H:%M:%S")}] imports done. Probing gates…', flush=True)

print()
print(f'  torch.__version__              = {torch.__version__}')
print(f'  torch.cuda.is_available()      = {torch.cuda.is_available()}')
print(f'  torch.cuda.get_device_name(0)  = {torch.cuda.get_device_name(0)}')
print(f'  occlude._BLUR_DEVICE           = {_BLUR_DEVICE}')
print(f'  torchcodec importable          = {_TORCHCODEC_OK}')
print(f'  ffmpeg has h264_nvenc          = {_ffmpeg_has_h264_nvenc()}')
print(f'  cuda_io_available()            = {cuda_io_available()}')
print(f'  default perception_batch       = {_default_perception_batch()}')
print()
assert _BLUR_DEVICE is not None and _BLUR_DEVICE.type == 'cuda', 'Step 1 CUDA blur dispatch is unavailable'
assert cuda_io_available(), 'Step 3+4 CUDA I/O unavailable — check torchcodec install + ffmpeg nvenc'
print('✓ Both fast paths armed.')

## 5. Unit tests (~10s)

In [ ]:
!cd /content/OCCLUDE && python -m pytest tests/ --tb=short

## 6. Test video

In [ ]:
!wget -q --show-progress -O /content/test_video.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/store-aisle-detection.mp4
!ls -lh /content/test_video.mp4
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,nb_frames -of default=nw=1 /content/test_video.mp4

## 6a. NVDEC + NVENC isolated smoke test

If §8 hangs at 0% GPU util, one of NVDEC or NVENC is stalling. This cell tests each independently with a hard timeout, so we know which side is broken before sinking minutes into a bench run that never finishes.

- **NVDEC test:** open the test video via `torchcodec.VideoDecoder(device='cuda')` and decode 1 frame.
- **NVENC test:** spawn an ffmpeg subprocess with `-c:v h264_nvenc`, write a single black frame, close stdin, check exit code.

If either hangs > 30 s the cell aborts with a clear error. Then jump to §8b (sync-fallback bench) for a Steps-1+2 result while we diagnose.

In [ ]:
import os, subprocess, time, threading
os.environ['PATH'] = '/opt/ffmpeg-nvenc/bin:' + os.environ['PATH']

def stamp(m): print(f'[{time.strftime("%H:%M:%S")}] {m}', flush=True)

# --- NVDEC ---
stamp('▶ NVDEC: opening torchcodec VideoDecoder on cuda…')
from torchcodec.decoders import VideoDecoder
import torch
result = {'frame': None, 'err': None}
def _decode_one():
    try:
        dec = VideoDecoder('/content/test_video.mp4', device='cuda')
        result['frame'] = next(iter(dec))
    except BaseException as e:
        result['err'] = e
t = threading.Thread(target=_decode_one, daemon=True)
t.start(); t.join(timeout=30)
if t.is_alive():
    raise RuntimeError('NVDEC hung > 30s — torchcodec CUDA decode is broken on this runtime. '
                       'Set OCCLUDE_DISABLE_CUDA_IO=1 to fall back to cv2 (Steps 1+2 only).')
if result['err'] is not None:
    raise result['err']
f = result['frame']
stamp(f'✓ NVDEC OK: frame shape={tuple(f.shape)} dtype={f.dtype} device={f.device}')

# --- NVENC: use communicate(input=...) so we don't double-close stdin ---
stamp('▶ NVENC: encoding a single 320×240 black frame with h264_nvenc…')
import numpy as np
W, H = 320, 240
frame = np.zeros((H, W, 3), dtype=np.uint8)
cmd = ['ffmpeg','-y','-loglevel','error','-f','rawvideo','-pix_fmt','bgr24',
       '-s', f'{W}x{H}', '-r', '30', '-i', '-', '-c:v', 'h264_nvenc', '-frames:v', '1',
       '/tmp/nvenc_test.mp4']
proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
try:
    _, err = proc.communicate(input=frame.tobytes(), timeout=30)
except subprocess.TimeoutExpired:
    proc.kill()
    raise RuntimeError('NVENC hung > 30s — h264_nvenc is unusable on this runtime. '
                       'Set OCCLUDE_DISABLE_CUDA_IO=1.')
if proc.returncode != 0:
    raise RuntimeError(f'NVENC encode failed (rc={proc.returncode}): {(err or b"").decode()[-500:]}')
size = os.path.getsize('/tmp/nvenc_test.mp4')
stamp(f'✓ NVENC OK: wrote /tmp/nvenc_test.mp4 ({size} bytes)')

# --- Full cuda_video_writer round-trip (the exact path the bench uses) ---
stamp('▶ cuda_video_writer round-trip: 30 frames through the bench writer context manager…')
from occlude.pipeline.io_cuda import cuda_video_writer
from pathlib import Path
roundtrip_path = Path('/tmp/cuda_writer_test.mp4')
result2 = {'err': None}
def _writer_round_trip():
    try:
        with cuda_video_writer(roundtrip_path, W, H, 30.0) as write:
            for i in range(30):
                f = np.full((H, W, 3), (i * 8) % 256, dtype=np.uint8)
                write(f)
    except BaseException as e:
        result2['err'] = e
t2 = threading.Thread(target=_writer_round_trip, daemon=True)
t2.start(); t2.join(timeout=30)
if t2.is_alive():
    raise RuntimeError('cuda_video_writer hung > 30s — the actual bench encode path is broken. '
                       'Set OCCLUDE_DISABLE_CUDA_IO=1 and re-run §8b.')
if result2['err'] is not None:
    raise result2['err']
stamp(f'✓ cuda_video_writer OK: {roundtrip_path} ({roundtrip_path.stat().st_size} bytes)')

print('\n✓ All three codec checks passed. §8 should not hang.')


## 7. Live progress GPU monitor

Run **this cell in the background** (click ▶, then click ▶ on §8). It prints GPU util + VRAM every 2s in this cell's output, so you can watch what the next cell's bench is actually doing on the GPU instead of staring at silent tqdm.

Stop it by hitting the ⬛ button on this cell when you're done with the bench.

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used,power.draw --format=csv -lms 2000

## 8. Bench — full overhaul, with stage logging

Calls the bench from Python (not `!shell`) so we can interleave timestamped progress prints. You'll see:

1. Model construction (~30s — downloads YOLO + SegFormer + InsightFace).
2. Warm-up call (`torch.compile` JIT, ~30-90s, GPU util will spike).
3. tqdm progress bar streaming frames.
4. Final bench line.

Each stage prints when it starts and when it finishes with elapsed seconds, so you always know what's running. Total ~3-5 min first time, ~1-2 min on repeat.

In [ ]:
import os, time, threading
from pathlib import Path

os.environ['PATH'] = '/opt/ffmpeg-nvenc/bin:' + os.environ['PATH']

import importlib
import occlude.pipeline.bench as bench_mod
import occlude.pipeline.video as video_mod
importlib.reload(bench_mod); importlib.reload(video_mod)

from occlude.pipeline.bench import run_benchmark, _GpuUtilSampler

INPUT = Path('/content/test_video.mp4')

def stamp(msg: str) -> None:
    print(f'[{time.strftime("%H:%M:%S")}] {msg}', flush=True)

stamp('▶ starting first run (downloads + JIT + 30s of frames)…')
t0 = time.perf_counter()
r1 = run_benchmark(INPUT, seconds=30.0)
stamp(f'✓ first run done in {time.perf_counter() - t0:.1f}s')
print('   →', r1.format_line())

print()
stamp('▶ starting second run (steady state, this is the H1 comparison)…')
t0 = time.perf_counter()
r2 = run_benchmark(INPUT, seconds=30.0)
stamp(f'✓ second run done in {time.perf_counter() - t0:.1f}s')
print('   →', r2.format_line())

print()
H1_FPS, H1_UTIL, H1_HASH = 8.34, 33.4, '396d6e6d1d175e8ef6b37088136f60ca'
speedup = r2.fps / H1_FPS
hash_ok = r2.frame_hash_md5 == H1_HASH
print('=' * 60)
print(f'  fps       {r2.fps:.2f}   ({speedup:.2f}× H1; target ≥3×)')
print(f'  util      {r2.gpu_util_avg:.1f}%   (target ≥70%, H1 33.4%)')
print(f'  VRAM      {r2.peak_vram_mb:.0f} MB   (H1 2660 MB)')
print(f'  hash      {r2.frame_hash_md5}')
print(f'  match H1? {"✓ yes" if hash_ok else "✗ DRIFT — investigate"}')
print('=' * 60)

## 8b. Fallback: bench with cuda_io disabled (Steps 1+2 only)

If §6a flagged NVDEC or NVENC as broken (or §8 hung), this cell forces the sync cv2 path while keeping Step 1's CUDA torch mask + Step 2's batched perception. You still get a fps improvement over H1 — just smaller than the full overhaul would deliver — and you confirm the rest of the pipeline is fine.

After this, send me §8b's output and we debug NVDEC/NVENC separately.

In [ ]:
import os, time, importlib
from pathlib import Path

os.environ['OCCLUDE_DISABLE_CUDA_IO'] = '1'   # ← the escape hatch
os.environ['PATH'] = '/opt/ffmpeg-nvenc/bin:' + os.environ['PATH']

# Force a re-import so the env var takes effect for already-loaded modules.
import occlude.pipeline.io_cuda as io_mod; importlib.reload(io_mod)
import occlude.pipeline.video as video_mod; importlib.reload(video_mod)
import occlude.pipeline.bench as bench_mod; importlib.reload(bench_mod)
from occlude.pipeline.bench import run_benchmark
from occlude.pipeline.io_cuda import cuda_io_available

print(f'cuda_io_available() = {cuda_io_available()}  (should be False)')
assert not cuda_io_available()

def stamp(m): print(f'[{time.strftime("%H:%M:%S")}] {m}', flush=True)

INPUT = Path('/content/test_video.mp4')
stamp('▶ first run (sync path, JIT + 30s of frames)…')
t0 = time.perf_counter()
r1 = run_benchmark(INPUT, seconds=30.0)
stamp(f'✓ done in {time.perf_counter() - t0:.1f}s')
print('   →', r1.format_line())

stamp('▶ second run (steady state)…')
t0 = time.perf_counter()
r2 = run_benchmark(INPUT, seconds=30.0)
stamp(f'✓ done in {time.perf_counter() - t0:.1f}s')
print('   →', r2.format_line())

print(f'\nspeedup over H1: {r2.fps / 8.34:.2f}× (this is Steps 1+2 only)')

## 9. Sensitivity sweep — find the perception-batch elbow

Each B value runs a full 30s bench; ~1-2 min apiece. Cell prints which B it's on, the elapsed time, and the resulting fps/util — no silent waits.

In [ ]:
import time
from occlude.pipeline.video import VideoProcessor
from occlude.pipeline.bench import run_benchmark, _GpuUtilSampler
import tempfile, torch
from pathlib import Path

INPUT = Path('/content/test_video.mp4')

def bench_with_batch(B: int, seconds: float = 30.0):
    """Inline copy of run_benchmark that takes perception_batch."""
    import cv2
    cap = cv2.VideoCapture(str(INPUT)); fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; cap.release()
    max_frames = max(1, int(round(fps * seconds)))
    with tempfile.TemporaryDirectory() as td:
        out = Path(td) / 'o.mp4'
        vp = VideoProcessor(perception_batch=B)
        torch.cuda.reset_peak_memory_stats()
        with _GpuUtilSampler() as s:
            t0 = time.perf_counter()
            vp.process(INPUT, out, max_frames=max_frames, skip_mux=True)
            wall = time.perf_counter() - t0
        peak = torch.cuda.max_memory_allocated() / (1024*1024)
    return wall, max_frames / wall if wall > 0 else 0.0, s.average, peak

results = []
for B in (1, 2, 4, 8):
    print(f'\n[{time.strftime("%H:%M:%S")}] ▶ B={B} …', flush=True)
    t0 = time.perf_counter()
    wall, fps, util, peak = bench_with_batch(B)
    elapsed = time.perf_counter() - t0
    util_s = f'{util:.1f}%' if util is not None else 'n/a'
    print(f'[{time.strftime("%H:%M:%S")}] ✓ B={B}: fps={fps:.2f} util={util_s} peak_vram={peak:.0f}MB (wall {wall:.1f}s)', flush=True)
    results.append((B, fps, util, peak))

print('\n' + '=' * 50)
print(f'{"B":>3} {"fps":>8} {"util":>8} {"vram":>8}')
for B, fps, util, peak in results:
    util_s = f'{util:.1f}%' if util is not None else 'n/a'
    print(f'{B:>3} {fps:>8.2f} {util_s:>8} {peak:>7.0f}MB')
print('=' * 50)

## 10. Interpretation

Send me §8's final summary block + §9's sweep table.

Hash drift = real regression (most likely `torch.compile` recompiling for new batch shapes and picking different kernels — fixable). fps under 25 = something didn't land; check the §4 gate output to see which fast path went unarmed.

If util is still well under 70% after this, the next bottleneck is the host-side `.cpu().numpy().tobytes()` in `cuda_video_writer` — a future step would move encode to a GPU-resident buffer.